## Seção 0 · Configuração do Ambiente

Instalação de dependências e configuração dos caminhos para execução local ou Google Colab.

In [ ]:
# @title 0.1 · Instalar dependências
!pip install -q sentence-transformers faiss-cpu rank-bm25 groq google-generativeai \
    python-dotenv tqdm pandas openai gradio

In [ ]:
# @title 0.2 · Configurar ambiente e API keys
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/pln-rag-normas-estruturais')
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = Path('.').resolve()
    if not (PROJECT_ROOT / 'src').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    IN_COLAB = False

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

SECTIONS_BASE_DIR = PROJECT_ROOT / 'data' / 'norms' / 'sections'
EVAL_DIR          = PROJECT_ROOT / 'data' / 'eval'
INDEX_DIR         = PROJECT_ROOT / 'index'

print(f"PROJECT_ROOT      : {PROJECT_ROOT}")
print(f"SECTIONS_BASE_DIR : {SECTIONS_BASE_DIR}")
print(f"GROQ_API_KEY      : {'✓' if os.getenv('GROQ_API_KEY') else '✗ — configure .env'}")


## Seção 1 · Ingestão das Normas

As normas NBR 6120, NBR 6118 e NBR 6123 foram convertidas para Markdown via Docling
e divididas em seções com frontmatter YAML (`title`, `summary`, `norm_id`, `edicao`)
usando o script `scripts/split_sections_auto.py`.

Cada seção fica em `data/norms/sections/<norm_id>/` e é uma unidade semântica auto-contida.
Para regenerar os arquivos:
```
python scripts/convert_pdfs.py          # PDF → Markdown
python scripts/split_sections_auto.py   # Markdown → seções
```


In [ ]:
# @title 1.1 · Listar seções existentes por norma
for norm_dir in sorted(SECTIONS_BASE_DIR.iterdir()):
    if not norm_dir.is_dir():
        continue
    files = sorted(norm_dir.glob('*.md'))
    print(f"\n[{norm_dir.name.upper()}] {len(files)} seções")
    for f in files:
        text = f.read_text(encoding='utf-8')
        title_line = [l for l in text.split('\n') if l.startswith('title:')]
        title = title_line[0].replace('title:', '').strip().strip('"') if title_line else f.stem
        print(f"  {f.name:55s} {len(text):6d} chars  |  {title[:50]}")


In [ ]:
# @title 1.2 · Carregar seções com metadados
from src.ingestion import load_sections, print_sections_summary

sections = load_sections(SECTIONS_BASE_DIR)
print_sections_summary(sections)


## Seção 2 · Chunking

Cada seção já é um chunk semântico — não há subdivisão adicional.
A granularidade seção→chunk garante que tabelas e parágrafos relacionados
permaneçam juntos, melhorando a precisão do retrieval.


In [ ]:
# @title 2.1 · Gerar chunks (1 chunk por seção)
from src.chunker import chunk_sections

chunks = chunk_sections(sections)

In [ ]:
# @title 2.2 · Inspecionar chunks
import pandas as pd

df_chunks = pd.DataFrame([
    {
        'chunk_id': c['chunk_id'],
        'secao': c['secao'],
        'summary': c['summary'][:60] + '...',
        'n_chars': c['n_chars'],
    }
    for c in chunks
])
print(df_chunks.to_string(index=False))
print(f"\nTotal: {len(chunks)} chunks | {df_chunks['n_chars'].sum():,} chars")

## Seção 3 · Embeddings e Índice FAISS

**Modelo:** `neuralmind/bert-base-portuguese-cased` (BERTimbau, dim=768)
— pré-treinado em português, sem necessidade de API externa.

**Índice:** `IndexFlatIP` com vetores L2-normalizados → busca exata por cosseno.


In [ ]:
# @title 3.1 · Carregar modelo de embedding
# ⏱️ Primeira execução: ~2-5 min (download ~400 MB)
from src.indexer import load_embedding_model

embed_model = load_embedding_model()

In [ ]:
# @title 3.2 · Construir e salvar índice FAISS
import time
from src.indexer import build_index, save_index

t0 = time.time()
faiss_index, indexed_chunks = build_index(chunks, model=embed_model)
save_index(faiss_index, indexed_chunks, index_dir=INDEX_DIR)
print(f"\n✅ Índice construído e salvo em {time.time()-t0:.1f}s")
print(f"   {faiss_index.ntotal} vetores · dim={faiss_index.d}")


## Seção 4 · Retrieval Denso (FAISS)

Busca semântica por similaridade de cosseno: a query é codificada pelo mesmo
modelo BERTimbau e o índice retorna os `k` chunks mais próximos.

**Parâmetros:**
- Modelo: `neuralmind/bert-base-portuguese-cased`, dim=768
- Índice: `IndexFlatIP` (busca exata), vetores L2-normalizados
- k: 3, 5 ou 10

In [ ]:
# @title 4.1 · Testar retrieval denso
from src.indexer import retrieve, print_retrieval_results

test_queries = [
    "Qual o valor de carga acidental para um pavimento de escritório?",
    "Quando é permitido reduzir as cargas acidentais em um edifício?",
    "Qual o peso específico do concreto armado?",
]

for q in test_queries:
    results = retrieve(q, faiss_index, indexed_chunks, embed_model, k=5)
    print_retrieval_results(results, q)

In [ ]:
# @title 4.2 · Comparar k=3, 5, 10 para a mesma query
q = "Qual a carga para dormitórios de edifícios residenciais?"
print(f"Query: {q}\n")
for k in [3, 5, 10]:
    results = retrieve(q, faiss_index, indexed_chunks, embed_model, k=k)
    print(f"  k={k}: {[r['chunk_id'] for r in results]}")

## Seção 5 · Pipeline RAG

Pipeline completo: Retrieval (FAISS) + Generation (Groq / NVIDIA NIM / Gemini).

O prompt instrui o LLM a responder **exclusivamente** com base nos trechos recuperados
e a citar explicitamente a seção normativa utilizada.

In [ ]:
# @title 5.1 · Inicializar pipeline RAG
# Provider: 'groq' (padrão) | 'nvidia' | 'gemini'
from src.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(provider='groq')
print("✅ Pipeline carregado.")

In [ ]:
# @title 5.2 · Testar pipeline RAG
questions = [
    "Qual o valor de carga acidental para escritórios?",
    "Como tratar paredes divisórias sem posição definida no projeto?",
    "Qual a redução de carga acidental quando há 6 ou mais pisos?",
]

for q in questions:
    res = pipeline.query(q, k=5, mode='dense')
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print(f"A: {res['answer']}")
    print(f"Fontes: {[s['chunk_id'] for s in res['sources']]}")
    print(f"Latência: retrieval={res['latency']['retrieval_s']:.2f}s · "
          f"geração={res['latency']['generation_s']:.2f}s")

## Seção 6 · Retrieval Esparso (BM25)

Busca léxica pura via **BM25Okapi** — sem embeddings, sem GPU.

**Parâmetros:**
- Tokenizador: regex alfanumérico, lowercase, sem stemming
- Score: BM25 (term frequency + inverse document frequency)

**Vantagem:** Palavras exatas (valores numéricos, nomes de materiais, códigos de seção)
são recuperadas com alta precisão.  
**Desvantagem:** Sem compreensão semântica — sinônimos e paráfrases não são capturados.


In [ ]:
# @title 6.1 · Criar e testar SparseRetriever (BM25)
from src.hybrid_search import SparseRetriever

sparse_retriever = SparseRetriever(indexed_chunks)

q = "Qual o valor de carga acidental para um pavimento de escritório?"
print(f"Query: {q}\n")
results_sparse = sparse_retriever.retrieve(q, k=5)
for r in results_sparse:
    print(f"  #{r['rank']} {r['chunk_id']:45s} BM25={r['score']:.3f}")

## Seção 7 · Retrieval Híbrido (BM25 + FAISS via RRF)

Combina os dois retrievers via **Reciprocal Rank Fusion (RRF)**:

```
score_RRF(chunk) = Σ 1 / (60 + rank_retriever)
```

**k_RRF = 60** (constante padrão da literatura).

**Deduplicação:** cada `chunk_id` acumula contribuições de ambos os retrievers
em um único score — sem repetições no resultado final.

**Candidatos:** `min(k×3, n_chunks)` por retriever antes da fusão.

In [ ]:
# @title 7.1 · Criar e testar HybridRetriever (BM25 + FAISS + RRF)
from src.hybrid_search import HybridRetriever

hybrid_retriever = HybridRetriever(indexed_chunks, faiss_index, embed_model)

q = "Qual o valor de carga acidental para um pavimento de escritório?"
print(f"Query: {q}\n")
results_hybrid = hybrid_retriever.retrieve(q, k=5)
for r in results_hybrid:
    print(f"  #{r['rank']} {r['chunk_id']:45s} RRF={r['score']:.5f}")

In [ ]:
# @title 7.2 · Comparar dense vs sparse vs hybrid numa mesma query
q = "Qual o peso específico do concreto armado?"
k = 5

print(f"Query: {q}  |  k={k}\n")
print(f"{'Modo':<8} {'#1':30s} {'#2':30s} {'#3':30s}")
print("-" * 100)

for mode_name, fn in [
    ('dense',  lambda q, k: retrieve(q, faiss_index, indexed_chunks, embed_model, k=k)),
    ('sparse', lambda q, k: sparse_retriever.retrieve(q, k=k)),
    ('hybrid', lambda q, k: hybrid_retriever.retrieve(q, k=k)),
]:
    res = fn(q, k)
    ids = [r['chunk_id'].replace('NBR6120#', '') for r in res[:3]]
    print(f"{mode_name:<8} {ids[0]:30s} {ids[1]:30s} {ids[2]:30s}")

## Seção 8 · Avaliação Comparativa Recall@k

**Recall@k**: para cada pergunta do golden set, verifica se o chunk correto
está entre os top-k recuperados. Calcula para k=3, 5, 10 nos **3 modos**.

**Golden set:** 21 perguntas sobre **NBR 6120** e **NBR 6123**, com evidências mapeadas
para chunk_ids exatos. Distribuição:
- 17 perguntas factuais diretas
- 3 perguntas multi-trecho (exigem combinar 2+ seções)
- 1 pergunta fora do corpus (teste de recusa)


In [ ]:
# @title 8.1 · Carregar e inspecionar golden set
from src.evaluator import load_golden_set
import pandas as pd

golden_set = load_golden_set(EVAL_DIR / 'golden_set.json')

df_gs = pd.DataFrame(golden_set)
print(df_gs[['id', 'categoria', 'pergunta', 'evidencia_esperada']].to_string(index=False))
print(f"\nTotal: {len(golden_set)} perguntas")
print(f"Categorias: {df_gs['categoria'].value_counts().to_dict()}")

In [ ]:
# @title 8.2 · Avaliação Recall@k — dense vs sparse vs hybrid
# ⏱️ < 5s para 21 perguntas × 13 chunks
import time
from src.evaluator import run_comparative_evaluation, print_comparative_report

def dense_fn(query, k):
    return retrieve(query, faiss_index, indexed_chunks, embed_model, k=k)

def sparse_fn(query, k):
    return sparse_retriever.retrieve(query, k=k)

def hybrid_fn(query, k):
    return hybrid_retriever.retrieve(query, k=k)

t0 = time.time()
comp_results = run_comparative_evaluation(
    dense_fn, sparse_fn, hybrid_fn, golden_set
)
print(f"\n✅ Avaliação concluída em {time.time()-t0:.1f}s")
print_comparative_report(comp_results)

In [ ]:
# @title 8.3 · Salvar relatório e analisar gaps
from src.evaluator import save_evaluation_report

save_evaluation_report(comp_results['modes']['dense'], output_path=INDEX_DIR)

comparison = comp_results['comparison']
print("\n📊 COMPARAÇÃO RECALL@K — dense vs sparse vs hybrid\n")
print(comparison.to_string(index=False))

dense_detail  = comp_results['modes']['dense']['details_df']
hybrid_detail = comp_results['modes']['hybrid']['details_df']
merged = dense_detail[['id', 'pergunta', 'hit@5']].merge(
    hybrid_detail[['id', 'hit@5']].rename(columns={'hit@5': 'hybrid_hit@5'}),
    on='id'
)
hybrid_gains = merged[(merged['hit@5'] == False) & (merged['hybrid_hit@5'] == True)]
print(f"\n🔼 Hybrid ganha sobre Dense @k=5: {len(hybrid_gains)} perguntas")
if len(hybrid_gains):
    print(hybrid_gains[['id', 'pergunta']].to_string(index=False))

## Seção 9 · Análise de Trade-offs: Dense vs Sparse vs Hybrid

### Quando cada modo se sai melhor

| Modo | Vantagem | Limitação |
|------|----------|-----------|
| **Dense** | Consultas semânticas (sinônimos, paráfrases) | Requer GPU/CPU para encoding; falha em termos muito específicos não vistos no treino |
| **Sparse** | Termos exatos: valores numéricos (`kN/m²`), nomes de materiais, códigos de seção (`2.2.1.3`) | Sem compreensão semântica — sinônimos e paráfrases não são capturados |
| **Hybrid** | Melhor dos dois: recupera por semântica E por termos exatos via RRF | Score RRF não é interpretável como probabilidade; ligeiramente mais lento que sparse puro |

### Impacto com o corpus atual (290 seções)

Com o corpus completo (NBR 6120:2019 + NBR 6123:2023), as diferenças entre os modos ficam mais evidentes.
As diferenças aparecem em k=3:

- **Dense** se sai melhor em perguntas paráfrasticas (ex.: "peso próprio da estrutura"
  recupera a seção de "ações permanentes" mesmo sem overlap léxico exato).
- **Sparse** se sai melhor em perguntas com valores numéricos exatos
  ("3 kN/m²", "25 kN/m³") ou referências a parágrafos ("§ 6.2", "§ 5.3").
- **Hybrid** normalmente empata com o melhor dos dois ou supera ambos.

### Latência (local, 290 chunks)

| Modo | Retrieval estimado | Gargalo |
|------|--------------------|---------|
| Sparse (BM25) | < 1 ms | — |
| Dense (FAISS) | 5–20 ms | Encoding da query com BERTimbau |
| Hybrid (RRF) | 5–20 ms | Dominado pelo encoding FAISS |

> Em produção com corpus >10k chunks, a vantagem de latência do Sparse fica evidente.
> Com corpora menores, todos os modos são praticamente instantâneos.

In [ ]:
# @title 9.1 · Benchmark de latência — dense vs sparse vs hybrid
import time

q = "Qual o valor de carga acidental para um pavimento de escritório?"
N_RUNS = 20

print(f"Query: {q}")
print(f"N_RUNS: {N_RUNS}\n")

results_lat = {}
for mode_name, fn in [
    ('dense',  lambda q: retrieve(q, faiss_index, indexed_chunks, embed_model, k=5)),
    ('sparse', lambda q: sparse_retriever.retrieve(q, k=5)),
    ('hybrid', lambda q: hybrid_retriever.retrieve(q, k=5)),
]:
    times = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        fn(q)
        times.append((time.perf_counter() - t0) * 1000)
    results_lat[mode_name] = times

print(f"{'Modo':<10} {'Média (ms)':>12} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 44)
for mode_name, times in results_lat.items():
    print(f"{mode_name:<10} {sum(times)/len(times):>12.1f} {min(times):>10.1f} {max(times):>10.1f}")

## Seção 10 · Interface Interativa (Gradio)

Interface completa diretamente no notebook — usa as variáveis já carregadas
(`pipeline`, `sparse_retriever`, `hybrid_retriever`). No Colab, use `demo.launch(share=True)`
para obter uma URL pública temporária.

In [ ]:
# @title 10.1 · Interface Gradio
import gradio as gr

def responder(pergunta, modo, k):
    if not pergunta.strip():
        return "Por favor, faça uma pergunta.", "", ""

    if modo == "hybrid":
        ret = hybrid_retriever
    elif modo == "sparse":
        ret = sparse_retriever
    else:
        ret = None

    res = pipeline.query(pergunta, k=int(k), mode=modo, retriever=ret)

    fontes = "\n\n".join([
        f"**#{s['rank']} · {s['chunk_id']}** (score: {s['score']:.3f})\n\n{s['texto'][:400]}..."
        for s in res['sources']
    ])

    lat = res['latency']
    info = (
        f"⏱ retrieval **{lat['retrieval_s']:.2f}s** · "
        f"geração **{lat['generation_s']:.2f}s** · "
        f"modo **{modo}** · k={k}"
    )

    return res['answer'], fontes, info

demo = gr.Interface(
    fn=responder,
    inputs=[
        gr.Textbox(
            label="Pergunta",
            lines=2,
            placeholder="Ex.: Qual o valor de carga acidental para escritórios?",
        ),
        gr.Radio(
            ["dense", "sparse", "hybrid"],
            label="Modo do retriever",
            value="dense",
        ),
        gr.Slider(minimum=3, maximum=10, step=1, value=5, label="Top-k"),
    ],
    outputs=[
        gr.Markdown(label="Resposta"),
        gr.Markdown(label="Trechos recuperados"),
        gr.Markdown(label="Latência"),
    ],
    title="RAG — NBR 6120 · NBR 6123",
    description="Chatbot técnico com citações normativas rastreáveis · NBR 6120:2019, NBR 6123:2023",
    examples=[
        ["Qual o valor típico de carga acidental para um pavimento de escritório?", "dense", 5],
        ["Qual o valor de carga acidental para uma garagem de veículos leves?", "dense", 5],
        ["O peso próprio da estrutura deve ser considerado como que tipo de carga?", "dense", 5],
        ["Como tratar paredes divisórias cuja posição não é definida no projeto?", "dense", 5],
        ["Qual o peso específico do concreto armado?", "dense", 5],
        ["Qual a carga que deve ser considerada ao longo de parapeitos e balcões?", "dense", 5],
        ["Quais critérios determinam a categoria de projeto para garagens e áreas de circulação de veículos?", "dense", 5],
        ["Qual a carga variável mínima a considerar em coberturas com acesso apenas para manutenção?", "dense", 5],
        ["Quando é permitido reduzir as cargas acidentais em um edifício?", "dense", 5],
        ["Qual a redução percentual de cargas acidentais quando há 6 ou mais pisos?", "hybrid", 5],
        ["Como é definida a velocidade básica do vento V0 pela NBR 6123?", "dense", 5],
        ["O que é a pressão dinâmica do vento e como é calculada?", "dense", 5],
        ["Quais são os três fatores que multiplicam V0 para obter a velocidade característica Vk?", "hybrid", 5],
        ["O que considera o fator topográfico S1 no cálculo do vento?", "dense", 5],
        ["O que considera o fator S2 no cálculo da velocidade do vento?", "dense", 5],
        ["Para uma residência normal, qual o valor mínimo do fator estatístico S3?", "dense", 5],
        ["O que são sobrepressão e sucção no contexto dos coeficientes de pressão do vento?", "dense", 5],
        ["Como é considerada a pressão interna do vento em edificações com aberturas?", "dense", 5],
        ["Em que situações a NBR 6123 indica o uso de ensaios em túnel de vento?", "dense", 5],
        ["Quando estruturas altas e esbeltas precisam considerar análise dinâmica além da análise estática?", "hybrid", 5],
        ["Como calcular o preço do m³ de concreto para uma obra em Brasília?", "dense", 5],
    ],
    flagging_mode="never",
)

demo.launch()  # Colab: demo.launch(share=True)